In [1]:
suppressPackageStartupMessages(library(SingleCellExperiment))
suppressPackageStartupMessages(library(scater))
suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(argparse))
suppressPackageStartupMessages(library(Seurat))

#####################
## Define settings ##
#####################
here::i_am("processing/1_create_seurat_rna.R")
source(here::here("settings.R"))
source(here::here("utils.R"))



here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/03_Stat3_RNA/code



In [2]:
args = list()
args$samples <- opts$samples
args$sce <- io$rna.sce
args$metadata <- "/rds/project/rds-SDzz0CATGms/users/bt392/03_Stat3_RNA/results/rna/mapping/sample_metadata_after_mapping.txt.gz"
# args$metadata <- paste0(io$basedir,"/results/rna/doublets/sample_metadata_after_doublets.txt.gz")
args$features <- 3000
args$npcs <- 40
args$n_neighbors = 30
args$min_dist = 0.3
args$test <- FALSE
args$colour_by <- c("celltype.mapped_mnn","sample")
args$vars.to.regress <- c("nFeature_RNA","mitochondrial_percent_RNA")
args$batch_correction <- c("stage")
args$remove_ExE_cells <- FALSE
args$outdir <- paste0(io$basedir,"/results/rna/mapping/test")

In [3]:
dir.create(args$outdir, recursive=TRUE, showWarnings = FALSE)

In [4]:
sample_metadata <- fread(args$metadata) %>%
   .[pass_rnaQC==TRUE & doublet_call==FALSE & sample%in%args$samples]

if (args$remove_ExE_cells) {
  print("Removing ExE cells...")
  sample_metadata <- sample_metadata %>%
    .[!celltype.mapped_mnn%in%c("Visceral_endoderm","ExE_endoderm","ExE_ectoderm","Parietal_endoderm")]
}

In [5]:
head(sample_metadata, 2)

cell,sample,barcode,nFeature_RNA,nCount_RNA,mitochondrial_percent_RNA,ribosomal_percent_RNA,stage,tdTom,pass_rnaQC,doublet_score,doublet_call,celltype.mapped_mnn,celltype.score_mnn,stage.mapped_mnn,cellstage.score_mnn,closest.cell_mnn,idx,tdTom_corr
<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<chr>,<lgl>,<lgl>,<dbl>,<lgl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<int>,<lgl>
SLX-21143_SITTA2_HTJH3DSX2#AAACCCAAGATGTTCC-1,SLX-21143_SITTA2_HTJH3DSX2,AAACCCAAGATGTTCC-1,4837,53199,1.47,22.87,E8.5,TRUE,TRUE,0.14,FALSE,Erythroid,1,E8.5,0.52,cell_76646,1,TRUE
SLX-21143_SITTA2_HTJH3DSX2#AAACCCATCAGACCTA-1,SLX-21143_SITTA2_HTJH3DSX2,AAACCCATCAGACCTA-1,4973,24165,0.95,19.26,E8.5,TRUE,TRUE,0.24,FALSE,Mesenchyme,1,E8.5,0.44,ext_cell_263969,3,TRUE


In [6]:
###################
## Sanity checks ##
###################

stopifnot(args$colour_by %in% colnames(sample_metadata))
# stopifnot(unique(sample_metadata$celltype.mapped) %in% names(opts$celltype.colors))

 if (length(args$batch_correction)>0) {
   stopifnot(args$batch_correction%in%colnames(sample_metadata))
   if (length(unique(sample_metadata[[args$batch_correction]]))==1) {
     message(sprintf("There is a single level for %s, no batch correction applied",args$batch_correction))
     args$batch_correction <- NULL
   } else {
     library(batchelor)
   }
 }

 if (length(args$vars_to_regress)>0) {
  stopifnot(args$vars_to_regress%in%colnames(sample_metadata))
 }

In [7]:

###############
## Load data ##
###############

# Load RNA expression data as SingleCellExperiment object
sce <- load_SingleCellExperiment(args$sce, cells=sample_metadata$cell, normalise = TRUE)

# Add sample metadata as colData
colData(sce) <- sample_metadata %>% tibble::column_to_rownames("cell") %>% DataFrame


In [8]:
seurat = as.Seurat(sce)

In [9]:
seurat

An object of class Seurat 
29453 features across 45667 samples within 1 assay 
Active assay: RNA (29453 features, 0 variable features)

In [10]:
rm(sce)

In [11]:
################
## Load Atlas ##
################

atlas = readRDS('/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended/embryo_sce.rds' )

In [12]:
# Rename ensemble IDs to gene names in the atlas
gene_metadata <- fread(io$gene_metadata) %>% .[,c("chr","ens_id","symbol")] %>%
  .[symbol!="" & ens_id%in%rownames(atlas)] %>%
  .[!duplicated(symbol)]

atlas <- atlas[rownames(atlas)%in%gene_metadata$ens_id,]
foo <- gene_metadata$symbol; names(foo) <- gene_metadata$ens_id
rownames(atlas) <- foo[rownames(atlas)]

In [13]:
atlas_meta = as.data.table(colData(atlas), keep.rownames=F)

Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”


In [14]:
atlas = atlas[, as.character(atlas_meta[!stage %in% c('-', 'E6.5', 'E6.75', 'E7.0', 'E7.25'), cell])]

In [15]:
atlas

class: SingleCellExperiment 
dim: 27561 389039 
metadata(0):
assays(1): counts
rownames(27561): Xkr4 Gm1992 ... mt-Nd6 mt-Cytb
rowData names(0):
colnames(389039): cell_361 cell_362 ... ext_cell_351871 ext_cell_351872
colData names(19): cell sample ... celltype_extended_atlas sizeFactor
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):

In [16]:
# Lognormalise
atlas <- logNormCounts(atlas,
                      size.factors = as.numeric(sizeFactors(atlas)))

In [17]:
atlas = as.Seurat(atlas)

In [18]:
# Multi core using future - built in to seurat
plan("multicore", workers = 24)
options(future.globals.maxSize = 50 * 1024 ^ 3) # for 50 Gb RAM

In [19]:
seurat.list = lapply(X = list(seurat, atlas), FUN = function(x) {
    x <- NormalizeData(x)
    x <- FindVariableFeatures(x, selection.method = "vst", nfeatures = args$features)
})

In [20]:
features <- SelectIntegrationFeatures(object.list = seurat.list)

In [21]:
head(features, 100)

[1] "Actc1"   "Myl7"    "Pf4"     "Myl4"    "Ttn"     "Rbp4"    "Ttr"    
  [8] "Afp"     "Tnnt2"   "Apoa1"   "Spink1"  "Myl3"    "Hbb-y"   "Tnnc1"  
 [15] "Hbb-bs"  "S100g"   "Hba-a1"  "Acta2"   "Apom"    "Apob"    "Myh6"   
 [22] "Nppa"    "Acta1"   "Hba-x"   "Hbb-bh1" "Ankrd1"  "Myh7"    "Aldh1a3"
 [29] "Apoa2"   "Myl2"    "Trh"     "Hba-a2"  "Lyve1"   "Rhox5"   "Csrp3"  
 [36] "Nepn"    "Col1a1"  "Sst"     "Apoe"    "Spp2"    "Cck"     "Tagln"  
 [43] "C1qb"    "Phlda2"  "Apoa4"   "Barx1"   "Col4a1"  "Lum"     "Amn"    
 [50] "Tnni1"   "Nefm"    "Postn"   "Lgals2"  "Hspb1"   "Etv2"    "Fabp3"  
 [57] "Ctsh"    "Dppa3"   "Sparc"   "Tnni3"   "Pyy"     "Sh3bgr"  "Cubn"   
 [64] "Prtn3"   "Crabp1"  "Tyrobp"  "Mt1"     "Fxyd2"   "Mesp1"   "Fgb"    
 [71] "Ripply2" "Nkx2-9"  "Tubb3"   "Trap1a"  "Cldn5"   "Col1a2"  "Col3a1" 
 [78] "Emcn"    "Nebl"    "Myog"    "Clec1b"  "Fcer1g"  "Phox2b"  "Trf"    
 [85] "Blvrb"   "Rhox9"   "Ccl3"    "Cdkn1c"  "Pgam2"   "Ppbp"    "Col4a2" 
 [92] "Ctla2a"  "Lefty2"  "Myl9"    "Plek"    "Srgn"    "Car2"    "Tdgf1"  
 [99] "Car4"    "Tac2"

In [22]:
seurat.list <- lapply(X = seurat.list, FUN = function(x) {
    x <- ScaleData(x, features = features, verbose = FALSE)
    x <- RunPCA(x, features = features, verbose = FALSE)
})

In [23]:
# # Multi core using future - built in to seurat
# plan("multicore", workers = 1)
# options(future.globals.maxSize = 50 * 1024 ^ 3) # for 50 Gb RAM

In [ ]:
anchors <- FindIntegrationAnchors(object.list = seurat.list, anchor.features = features, reduction = "rpca")

Scaling features for provided objects

Computing within dataset neighborhoods

Finding all pairwise anchors



In [ ]:
# this command creates an 'integrated' data assay
seurat.integrated <- IntegrateData(anchorset = anchors)

In [ ]:
seurat.integrated

In [ ]:
# specify that we will perform downstream analysis on the corrected data note that the
# original unmodified data still resides in the 'RNA' assay
DefaultAssay(seurat.integrated) <- "integrated"

# Run the standard workflow for visualization and clustering
seurat.integrated <- ScaleData(seurat.integrated, verbose = FALSE)
seurat.integrated <- RunPCA(seurat.integrated, npcs = 30, verbose = FALSE)
seurat.integrated <- RunUMAP(seurat.integrated, reduction = "pca", dims = 1:30)